In [1]:
import pytest
import ipytest
from elements import *
ipytest.autoconfig()

In [2]:
%%ipytest


def test_elements_initialization():
    # Test single position (1D array) conversion to (1, 3)
    e = Elements(r=[1, 2, 3])
    assert e.r.shape == (1, 3)
    assert e.N == 1
    # Default orientation should be [0, 0, 1]
    assert np.array_equal(e.n, [[0, 0, 1]])

def test_optical_tx_initialization():
    # Test that subclasses correctly inherit and broadcast scalar properties
    r = [[0, 0, 0], [1, 1, 1]]
    tx = OpticalTxElements(r=r, p=10)
    assert tx.N == 2
    assert tx.p.shape == (2, 1)
    assert np.all(tx.p == 10)

# --- Test Addition (__add__) ---

def test_elements_addition():
    e1 = Elements(r=[0, 0, 0], n=[0, 0, 1])
    e2 = Elements(r=[1, 1, 1], n=[0, 0, 1])
    
    combined = e1 + e2
    
    assert combined.N == 2
    assert combined.r.shape == (2, 3)
    assert np.array_equal(combined.r[1], [1, 1, 1])

def test_addition_type_safety():
    tx = OpticalTxElements(r=[0, 0, 0])
    rx = OpticalRxElements(r=[1, 1, 1])
    
    # Adding different types should raise TypeError
    with pytest.raises(TypeError):
        _ = tx + rx

def test_addition_mismatched_optional_fields():
    rx1 = OpticalRxElements(r=[0, 0, 0], refl=0.5)
    rx2 = OpticalRxElements(r=[1, 1, 1], refl=None)
    
    # Adding one with 'refl' and one without should raise ValueError
    with pytest.raises(ValueError, match="Field 'refl' is None in one object"):
        _ = rx1 + rx2

# --- Test Merging (merge) ---

def test_merge_multiple_batches():
    batches = [
        OpticalTxElements(r=[0, 0, 0], p=1),
        OpticalTxElements(r=[1, 1, 1], p=2),
        OpticalTxElements(r=[2, 2, 2], p=3)
    ]
    
    merged = OpticalTxElements.merge(batches)
    
    assert merged.N == 3
    assert np.array_equal(merged.p.flatten(), [1, 2, 3])

def test_merge_with_none_padding():
    # Test the logic where 'refl' is None in some batches but not others
    rx1 = OpticalRxElements(r=[0, 0, 0], refl=0.8) # N=1
    rx2 = OpticalRxElements(r=[[1, 1, 1], [2, 2, 2]], refl=None) # N=2
    
    merged = OpticalRxElements.merge([rx1, rx2])
    
    assert merged.N == 3
    # First element should be 0.8, others should be padded with 0.0
    assert merged.refl[0] == 0.8
    assert merged.refl[1] == 0.0
    assert merged.refl[2] == 0.0

def test_merge_empty_list():
    assert Elements.merge([]) is None

........                                                                                     [100%]
8 passed in 0.03s
